In [1]:
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta
import pandas as pd
#  Convert ENTSO-E XML to DataFrame
def parse_response(xml_content):
    root = ET.fromstring(xml_content)
    ns = {"ns": "urn:iec62325.351:tc57wg16:451-7:actualgeneration:0:6"}
    data = []

    for time_series in root.findall("ns:TimeSeries", ns):
        production_type = time_series.find("ns:MktPSRType/ns:psrType", ns).text
        period = time_series.find("ns:Period", ns)
        start_time = period.find("ns:timeInterval/ns:start", ns).text
        resolution = period.find("ns:resolution", ns).text
        points = period.findall("ns:Point", ns)
        
        for point in points:
            position = int(point.find("ns:position", ns).text)
            quantity = float(point.find("ns:quantity", ns).text)
            timestamp = datetime.fromisoformat(start_time) + timedelta(hours=position - 1)
            data.append({
                "datetime": timestamp,
                "production_type": production_type,
                "quantity_MW": quantity
            })
    return pd.DataFrame(data)
